In [ ]:
## setup

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import CACHE_DIR, FIG_DIR, SEQ_LEN
from src.data_loader import build_area_series

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)

# Reload EDA outputs produced by notebook 01
agg_path = CACHE_DIR / "total_traffic_per_area.parquet"
assert agg_path.exists(), "Run notebook 01 (Task 2.1) first."

total_traffic = (pd.read_parquet(agg_path)
                    .set_index("Square id")["total_traffic"]
                    .sort_values(ascending=False))
top3 = total_traffic.head(3).index.tolist()
highest = top3[0]
print(f"Top-3 areas: {top3}  |  Highest-traffic area: {highest}")

In [ ]:
## EDA

print("=" * 60)
print("EDA EVIDENCE FOR MODEL SELECTION")
print("=" * 60)

print(f"\nAreas with traffic: {len(total_traffic):,}")
print(f"Distribution skewness: {total_traffic.skew():.2f}")
print(f"Top 1% traffic share:  "
      f"{total_traffic.head(max(1, int(len(total_traffic)*0.01))).sum() / total_traffic.sum() * 100:.1f}%")

print(f"\nHighest-traffic area (Square {highest}):")
print(f"  Mean: {total_traffic.iloc[0]:,.0f} CDRs over ~2 months")
print(f"  Daily seasonality: period = 144 intervals (24 h × 6)")
print(f"  Strong ACF peaks at lag 144 and 288 (see Task 2.3)")
print(f"  STL: daily seasonality dominates the variance")
print(f"  ADF/KPSS: difference-stationary with strong seasonality")

In [ ]:
## Comparison table of top-3 areas

model_table = pd.DataFrame([
    {
        "Model": "SARIMA",
        "Family": "Classical statistical (state-space)",
        "Input representation": "Univariate series, seasonal period s=144",
        "Why selected": (
            "Daily seasonality dominates the highest-traffic areas; "
            "SARIMA is the natural linear baseline with explicit "
            "seasonal AR/MA structure. Ferreira et al. (2023) and "
            "Hussien et al. (2025) both use SARIMA as the statistical "
            "reference point."
        ),
        "Expected strengths": (
            "Interpretable parameters; strong on clean daily cycles; "
            "small computational footprint."
        ),
        "Expected limitations": (
            "Cannot represent sharp non-linear bursts; expensive refit "
            "per forecast step; assumes fixed seasonal pattern."
        ),
    },
    {
        "Model": "LSTM",
        "Family": "Deep recurrent neural network",
        "Input representation": f"Univariate window of {SEQ_LEN} (2 days), z-score normalised",
        "Why selected": (
            "ACF shows long-range dependencies (lag 1008 still "
            "correlated); LSTM gating captures long temporal context. "
            "Azari et al. (2019) and Shindou et al. (2025) show LSTM "
            "outperforms ARIMA on non-linear traffic."
        ),
        "Expected strengths": (
            "Captures non-linear dynamics; learns long-range patterns "
            "automatically; scales to multivariate inputs."
        ),
        "Expected limitations": (
            "Slow training (~2–5 min per area); heavy hyperparameter "
            "sensitivity; black-box."
        ),
    },
    {
        "Model": "XGBoost",
        "Family": "Gradient-boosted decision trees",
        "Input representation": "Lag + rolling + cyclic time features",
        "Why selected": (
            "Hussien et al. (2025) found XGBoost was the best "
            "non-ensemble model on this exact Milan dataset. Its "
            "engineered features match the deterministic daily cycle "
            "observed in the EDA."
        ),
        "Expected strengths": (
            "Fast training; strong on tabular features; robust to "
            "non-stationarity; feature importances explain predictions."
        ),
        "Expected limitations": (
            "Lags dominate → one-step-late on abrupt bursts; needs "
            "manual feature engineering; cannot extrapolate trend."
        ),
    },
])

pd.set_option("display.max_colwidth", 80)
display(model_table.set_index("Model"))

In [ ]:
## model chech

# Load the highest-traffic area's series and plot 2 weeks.
# This confirms the daily cycle → justifies both SARIMA (s=144)
# and the cyclic features used in XGBoost.

ts = build_area_series(highest)
sample = ts.loc["2013-11-01":"2013-11-14"]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample.index, sample.values, linewidth=0.8, color="black")
ax.set_title(f"Square {highest} — first two weeks (highest-traffic area)")
ax.set_ylabel("Internet traffic")
ax.set_xlabel("Date")
plt.tight_layout()
plt.savefig(FIG_DIR / "model_selection_eda_confirmation.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# exporting

out = CACHE_DIR / "model_selection_table.parquet"
model_table.to_parquet(out)
print(f"Saved → {out}")

Literature summary. Ferreira et al. (2023) compare ARMA/ARIMA/SARIMA against RNN/LSTM/GRU/CNN and show no single family dominates. Hussien et al. (2025) evaluate eight models on this exact Milan dataset and find SARIMA weakest and an ensemble CNN+LSTM strongest (R² = 0.990), with XGBoost the best non-ensemble model. Shindou et al. (2025) find GRU most efficient, BiLSTM most accurate. Azari et al. (2019) show LSTM outperforms ARIMA on non-stationary segments.

Our choice. We select one model from each family — SARIMA (statistical), LSTM (recurrent), XGBoost (tree-based) — so the comparison is meaningful and spans the space of methods used in the literature. Our EDA (strong daily seasonality, long-range autocorrelation, difference-stationarity) supports all three: SARIMA for the seasonal cycle, LSTM for long-range non-linear context, XGBoost for engineered calendar and lag features.